# web_agent — Kaggle driver (hybrid)

Stable core (`labels`, `config`, `dataset`, `model`, `loss`, `metrics`) lives in the
`web_agent` package, cloned from GitHub. Orchestration + analysis (training loop,
eval, plots, error inspection, result images) lives **here in the notebook** so you
can watch every step.

Workflow: edit package in IDE → push to GitHub `Code` → **Pull** here → re-run.

In [ ]:
# 1. Clone the package + install deps, make it importable via sys.path (re-run safe)
#
# No `pip install -e .` (PEP 660 editable finder shadows web_agent.data). Plain
# sys.path. QLoRA needs peft + bitsandbytes + accelerate; class weights need sklearn.
import os, sys
REPO = "https://github.com/Kiyas-Mahmud/webagent.git"
ROOT = "/kaggle/working/webagent"
SRC = f"{ROOT}/src"
if not os.path.isdir(ROOT):
    !git clone -b Code {REPO} {ROOT}
else:
    !cd {ROOT} && git pull --ff-only
%cd {ROOT}

!pip install -q -U "transformers>=4.49" peft bitsandbytes accelerate scikit-learn

for m in [k for k in list(sys.modules) if k == "web_agent" or k.startswith("web_agent.")]:
    del sys.modules[m]
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import web_agent
from web_agent.data import verify_dataset
print("web_agent ready ->", list(web_agent.__path__))

In [ ]:
# 2. Confirm GPU + dataset path, then verify the dataset (Phase 1, in-kernel)
import os
from pathlib import Path
!nvidia-smi -L
print("input dirs:", os.listdir("/kaggle/input"))

DATA_PATH = "/kaggle/input/datasets/kiyasmahmud/thesisdata/FinalData"
assert os.path.isdir(DATA_PATH), f"fix DATA_PATH; not found: {DATA_PATH}"

from web_agent.data.verify_dataset import verify
report = verify(Path(DATA_PATH))
print(report.text())
assert report.ok, "dataset verification FAILED — read the report above"

In [ ]:
# 3. Phase 2/3 — Qwen2-VL processor + one VLM batch (joint image+text)
import torch
from transformers import AutoProcessor
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
from web_agent.data.dataloader import load_split, build_dataloader

set_seed(42)
cfg = load_config("configs/backbones/qwen2vl_2b.yaml")   # smaller/faster 2B, fp16
cfg["data"]["root"] = DATA_PATH
cfg["data"]["num_workers"] = 0          # VLM processor: avoid worker pickling overhead

bb = cfg["backbone"]
processor = AutoProcessor.from_pretrained(
    bb["vlm_model"], min_pixels=bb["min_pixels"], max_pixels=bb["max_pixels"],
)

train = load_split(cfg, "train")
loader = build_dataloader(cfg, mode="full_labels", records=train,
                          processor=processor, tokenizer=None,
                          limit=cfg["stages"]["smoke"],
                          batch_size=cfg["optim"]["batch_size"])

batch = next(iter(loader))
print("batch keys:", list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"  {k:22} {tuple(v.shape)}  {v.dtype}")
    else:
        print(f"  {k:22} {type(v).__name__} len={len(v)}  e.g. {v[0]!r}")

In [ ]:
# 4. Visual sanity-check — show 4 raw screenshots with their decoded labels
import matplotlib.pyplot as plt
from PIL import Image
from web_agent.labels import (
    EXECUTION_OUTCOME_INV, FAILURE_TYPE_INV, ACTION_TYPE_INV, RECOVERY_STRATEGY_INV,
)

rows = loader.dataset.records[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, rec in zip(axes, rows):
    ax.imshow(Image.open(f"{DATA_PATH}/{rec['state_before']}").convert("RGB"))
    ax.axis("off")
    ax.set_title(
        f"{rec['execution_outcome']} / {rec['failure_type']}\n"
        f"act={rec['action_type']}  rec={rec['recovery_strategy']}\n"
        f"conf={rec['agent_confidence_before']:.2f}  bbox={'Y' if rec['action_target_bbox'] else 'N'}",
        fontsize=9,
    )
plt.tight_layout(); plt.show()

# Confirm encoded labels match the strings above
print("encoded label_outcome :", batch["label_outcome"][:4].tolist())
print("encoded label_action  :", batch["label_action"][:4].tolist())
print("bbox_mask             :", batch["bbox_mask"][:4].squeeze(-1).tolist())

In [ ]:
# 5. Build model (Qwen2-VL-2B 4-bit QLoRA + adapter + 5 heads) + class-weighted 10-term loss
import torch
from web_agent.models.model import WebAgentModel
from web_agent.models.loss import CombinedLoss
from web_agent.utils.class_weights import balanced_class_weights, binary_pos_weight

# DIAGNOSTIC: isolate the outcome head to measure its ceiling (plan D1/D2/D3).
# When True, zero the other 9 loss terms so only outcome trains — compare MCC across
# pooling=mean|last|attention (cfg) and use_state_after=false|true to find the real
# wall. Set False for the full 10-term run.
DIAG_OUTCOME_ONLY = True
if DIAG_OUTCOME_ONLY:
    for k in ("failure_type", "action_type", "bbox", "memory_flag", "recovery",
              "confidence", "calibration", "contrastive", "recovery_outcome"):
        cfg["loss"][k] = 0.0
    print("DIAG: outcome-only loss; pooling =", cfg["backbone"].get("pooling"),
          "| use_state_after =", cfg["data"].get("use_state_after"))

model = WebAgentModel(cfg)          # 4-bit load + LoRA (prints trainable %), ~4GB download first run
print("VLM hidden dim D =", model.encoder.hidden_dim, "-> adapter -> 768",
      "| pooling =", cfg["backbone"].get("pooling", "last"))

device = "cuda"
for m in (model.adapter, model.failure_head, model.action_head,
          model.memory_head, model.recovery_outcome_head):
    m.to(device)

# action/failtype = sklearn 'balanced'; outcome = CAPPED (ratio<=1.5) to correct the
# imbalance WITHOUT the over-swing that just flips the collapse (plan P1-B).
aw, fw, ow = balanced_class_weights(train)          # outcome_scheme="capped" default
rsw = binary_pos_weight(train, "recovery_success")  # pos_weight for the recovery head (P2-B)
print("action weights  :", [round(x, 2) for x in aw.tolist()])
print("failtype weights:", [round(x, 2) for x in fw.tolist()])
print("outcome weights :", [round(x, 2) for x in ow.tolist()], "(capped)")
print("recovery pos_w  :", round(float(rsw), 2))
loss_fn = CombinedLoss(cfg, action_class_weights=aw.to(device),
                       failtype_class_weights=fw.to(device),
                       outcome_class_weights=ow.to(device),
                       recovery_success_pos_weight=rsw.to(device)).to(device)

print("trainable params:", f"{sum(p.numel() for p in model.trainable_parameters()):,}")

In [ ]:
# 6. SMOKE — 5 heads + 10 loss terms, contrastive non-zero, one LoRA step (pair-sampler batch)
# Rebuild loss_fn from a FRESH import so a stale cached module can't bite.
import importlib, web_agent.models.loss as _lossmod
importlib.reload(_lossmod)
loss_fn = _lossmod.CombinedLoss(cfg, action_class_weights=aw.to(device),
                                failtype_class_weights=fw.to(device),
                                outcome_class_weights=ow.to(device),
                                recovery_success_pos_weight=rsw.to(device)).to(device)

# batch_size 4: 2 images/sample (use_state_after) ~2x vision tokens -> fit T4.
smoke_loader, smoke_sampler = build_dataloader(
    cfg, "full_labels", train, processor, tokenizer=None,
    limit=64, batch_size=4, pair_sampler=True, num_workers=0)

opt = torch.optim.AdamW(model.trainable_parameters(), lr=cfg["optim"]["lr_heads"])
model.train()
lora_before = next(iter(model.lora_parameters())).detach().clone()

batch = next(iter(smoke_loader))
with torch.autocast("cuda", dtype=torch.float16):
    preds = model(batch)
    terms = loss_fn(preds, {k: (v.to(device) if torch.is_tensor(v) else v)
                            for k, v in batch.items()})

print("loss terms:", {k: round(float(v.detach()), 4) for k, v in terms.items()})
assert torch.isfinite(terms["total"]), "loss NaN/inf - STOP"
# SupCon contrastive non-zero only if this batch holds both outcomes (pair-sampler may not).
print("contrastive non-zero:", float(terms["contrastive"].detach()) != 0.0)

terms["total"].backward()
opt.step()
lora_after = next(iter(model.lora_parameters())).detach()
print("LoRA weight changed:", not torch.equal(lora_before, lora_after))
print("peak GPU mem (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))
print("SMOKE RESULT:", "PASS" if torch.isfinite(terms["total"]) else "FAIL")

# 7. MINI via Trainer — tiny train+val, 1 epoch: exercises the full loop + metrics CSV
import copy
from web_agent.train.trainer import Trainer

mini_cfg = copy.deepcopy(cfg)
mini_cfg["train"]["epochs"] = 1
mini_cfg["optim"]["grad_accum"] = 2

train_loader, train_sampler = build_dataloader(
    mini_cfg, "full_labels", train, processor, tokenizer=None,
    limit=400, batch_size=8, pair_sampler=True, num_workers=2)

val = load_split(mini_cfg, "val")
val_loader = build_dataloader(
    mini_cfg, "eval_labels", val, processor, tokenizer=None,
    limit=200, batch_size=8, shuffle=False, num_workers=2)

trainer = Trainer(model, loss_fn, mini_cfg, train_loader, val_loader,
                  train_sampler=train_sampler)
summary = trainer.fit()      # prints epoch metrics incl failure_f1; writes results CSV
print(summary)

In [ ]:
# 8. Checkpoint round-trip — save LoRA + adapter + 5 heads as ONE dict, reload, compare
from web_agent.utils.checkpoint import save_checkpoint, load_checkpoint
from peft import set_peft_model_state_dict

state = trainer._state()
save_checkpoint("checkpoints/qwen2vl2b_probe.ckpt", **state)

model.eval()
probe = next(iter(val_loader))
with torch.no_grad():
    before = model(probe)["outcome"].detach().cpu()

ck = load_checkpoint("checkpoints/qwen2vl2b_probe.ckpt")
set_peft_model_state_dict(model.encoder.model, ck["lora"])
model.adapter.load_state_dict(ck["adapter"])
model.failure_head.load_state_dict(ck["failure"])
model.action_head.load_state_dict(ck["action"])
model.memory_head.load_state_dict(ck["memory"])
model.recovery_outcome_head.load_state_dict(ck["recovery_outcome"])
with torch.no_grad():
    after = model(probe)["outcome"].detach().cpu()

print("checkpoint round-trip identical:", torch.allclose(before, after, atol=1e-5))

In [ ]:
# 9. TEST RUN (4k rows, 3 epochs) -> 3-split eval. Big enough to read the MCC TREND.
#    Inputs now = before+after images + task/target/domain + visual_change (cfg use_state_after).
#    Judge by outcome_mcc / outcome_bal_acc / success_recall PER EPOCH — NOT failure_f1.
#    Go/no-go: MCC trending up (0.05->0.15->0.25) -> scale to full run.
#             MCC stuck ~0 -> deeper data issue. Resumes via checkpoints/<name>/last.ckpt.
from web_agent.train.trainer import Trainer
from web_agent.eval.evaluate import evaluate_all_splits

TRAIN_ROWS = 4000
VAL_MON    = 1200
cfg["train"]["epochs"] = 3
cfg["data"]["num_workers"] = 4

# pair_sampler=False -> random batches so SupCon sees BOTH outcome classes per batch
# (the task-pair sampler makes batches single-outcome and starves the contrastive term).
full_train_loader = build_dataloader(
    cfg, "full_labels", train, processor, tokenizer=None, limit=TRAIN_ROWS,
    batch_size=cfg["optim"]["batch_size"], pair_sampler=False, shuffle=True, num_workers=4)

val_full = load_split(cfg, "val")
val_mon_loader = build_dataloader(
    cfg, "eval_labels", val_full, processor, tokenizer=None, limit=VAL_MON,
    batch_size=cfg["optim"]["batch_size"], shuffle=False, num_workers=4)

trainer = Trainer(model, loss_fn, cfg, full_train_loader, val_mon_loader,
                  train_sampler=None)
summary = trainer.fit()             # live per-step loss+ETA; top-3 ckpts; metrics CSV
print("training done:", summary)

# FINAL evaluation on the full 3 generalization splits (test_task / website / domain).
results = evaluate_all_splits(model, cfg, processor, device=device)
for split, m in results.items():
    print(split, {k: round(v, 4) for k, v in m.items()})